# Gamma Knife — прогнозирование интракраниальной прогрессии

Соревновательное решение (Kaggle, `gamma-knife-3`): ансамбль **CatBoost + LightGBM + XGBoost** с блендингом по OOF-предсказаниям.

**Пайплайн:** импорты → загрузка данных → EDA и предобработка → генерация признаков → обучение трёх моделей (OOF) → подбор весов и порога блендинга → сабмиты.

> Метрика: balanced accuracy (итог блендинга ≈ 0.886). Клинические данные пациентов в репозиторий не включены.

# Импорты

In [ ]:
import torch

In [ ]:
!pip install catboost
!pip install optuna
!pip install category_encoders
!pip install xgboost
!pip install lightgbm
!pip install imbalanced-learn

In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, fbeta_score, roc_auc_score, classification_report, precision_recall_curve, roc_curve, roc_auc_score, precision_recall_curve, auc
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import VotingClassifier, StackingClassifier
from optuna.pruners import SuccessiveHalvingPruner
import lightgbm as lgb
import category_encoders as ce
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

In [ ]:
!pip install google

In [ ]:
!pip install kaggle

In [ ]:
!kaggle competitions download -c gamma-knife-3

In [ ]:
import zipfile
import os

zip_path = "gamma-knife-3.zip"
extract_path = "gamma-knife-3"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Готово, распаковано в папку:", extract_path)

# EDA

## EDA и предобработка
Загрузка train/test, приведение чисел (запятая→точка), парсинг дат, формирование целевой переменной (`Прогрессия`), бинарные флаги процедур, заполнение категорий `Unknown`.

In [ ]:
df = pd.read_csv("gamma-knife-3/train.csv")
df_test = pd.read_csv("gamma-knife-3/test.csv")
df.head()

df_test['Объем максимального очага'] = df_test['Объем максимального очага'].str.replace(',', '.').astype('float')
df_test['Суммарный объем очагов'] = df_test['Суммарный объем очагов'].str.replace(',', '.').astype('float')
df_test.head()

In [ ]:
dates = ['Дата рождения', 'Дата постановки онкологического диагноза / начала первичного лечения', 'Дата удаления первичного очага', 'Дата развития МГМ', 'Дата проведения ОВГМ', 'Дата операции на ГМ', 'Дата 1-ой РХ']

for date in dates:
  df[date] = pd.to_datetime(df[date], format = '%d.%m.%Y', errors = 'coerce')
  df_test[date] = pd.to_datetime(df_test[date], format = '%d.%m.%Y', errors = 'coerce')

In [ ]:
df['Пол'].value_counts()
df['Пол'] = df['Пол'].replace('м', 'М').replace('ж', 'Ж')

df['Мужчина'] = df['Пол'].apply(lambda x: 1 if x == 'М' else 0)

df.drop(columns = ['Пол'], axis = 1, inplace = True)

#Для итоговых данных

df_test['Пол'].value_counts()
df_test['Пол'] = df_test['Пол'].replace('м', 'М').replace('ж', 'Ж')

df_test['Мужчина'] = df_test['Пол'].apply(lambda x: 1 if x == 'М' else 0)

df_test.drop(columns = ['Пол'], axis = 1, inplace = True)

In [ ]:
df['Возраст'] = (df['Дата 1-ой РХ'] - df['Дата рождения']).dt.days / 365.25
df['Время_метастазирования'] = (df['Дата развития МГМ'] - df['Дата постановки онкологического диагноза / начала первичного лечения']).dt.days
df['Время_реагирования'] = (df['Дата 1-ой РХ'] - df['Дата развития МГМ']).dt.days

df['ОВГМ'] = df['Дата проведения ОВГМ'].notna().astype(int)
df['Операция'] = df['Дата операции на ГМ'].notna().astype(int)

df.drop(columns = ['Дата рождения', 'Дата развития МГМ', 'Дата постановки онкологического диагноза / начала первичного лечения', 'Дата проведения ОВГМ', 'Дата операции на ГМ', 'Дата 1-ой РХ', 'Дата удаления первичного очага'], axis = 1, inplace = True)

#Для итоговых данных

df_test['Возраст'] = (df_test['Дата 1-ой РХ'] - df_test['Дата рождения']).dt.days / 365.25
df_test['Время_метастазирования'] = (df_test['Дата развития МГМ'] - df_test['Дата постановки онкологического диагноза / начала первичного лечения']).dt.days
df_test['Время_реагирования'] = (df_test['Дата 1-ой РХ'] - df_test['Дата развития МГМ']).dt.days

df_test['ОВГМ'] = df_test['Дата проведения ОВГМ'].notna().astype(int)
df_test['Операция'] = df_test['Дата операции на ГМ'].notna().astype(int)

df_test.drop(columns = ['Дата рождения', 'Дата развития МГМ', 'Дата постановки онкологического диагноза / начала первичного лечения', 'Дата проведения ОВГМ', 'Дата операции на ГМ', 'Дата 1-ой РХ', 'Дата удаления первичного очага'], axis = 1, inplace = True)

In [ ]:
# Оставляем только строки с известной целевой переменной
df = df[df['Интракраниальная прогрессия'].notna()]

# Таргет
df['Прогрессия'] = df['Интракраниальная прогрессия'].apply(
    lambda x: 1 if x in ['ДМ', 'ЛР+ДМ', 'ЛР'] else 0
)

df.drop(columns=['Интракраниальная прогрессия', 'Активирующие мутации'], inplace=True)


def create_procedure_flag(series):
    """
    Бинарный флаг события/процедуры:
    - 1 если значение задано и это НЕ 'нет'
      (например дата или любое другое "да/есть/..." значение)
    - 0 если NaN/пусто или явно 'нет'
    """
    s = series.astype(str).str.strip().str.lower()

    # где было NaN, после astype(str) станет 'nan' — считаем это отсутствием
    is_missing = s.isin(['nan', 'none', ''])
    is_no = s.eq('нет')

    return (~is_missing & ~is_no).astype(int)


# train
df['Экстракраниальные метастазы'] = create_procedure_flag(df['Экстракраниальные метастазы'])
df['Дистантные метастазы'] = create_procedure_flag(df['Дистантные метастазы'])
df['Локальный рецидив'] = create_procedure_flag(df['Локальный рецидив'])

# test
df_test.drop(columns=['Активирующие мутации'], inplace=True)
df_test['Экстракраниальные метастазы'] = create_procedure_flag(df_test['Экстракраниальные метастазы'])
df_test['Дистантные метастазы'] = create_procedure_flag(df_test['Дистантные метастастазы']) if 'Дистантные метастастазы' in df_test.columns else df_test.get('Дистантные метастазы')
df_test['Локальный рецидив'] = create_procedure_flag(df_test['Локальный рецидив']) if 'Локальный рецидив' in df_test.columns else df_test.get('Локальный рецидив')


In [ ]:
# OHE = ['Онкологический диагноз', 'Лекарственное лечение']
# df_encoded = pd.get_dummies(df, columns = OHE, prefix = '', prefix_sep='', dtype = 'int')
df_encoded = df.copy()

In [ ]:
df_encoded['Суммарный объем очагов'] = df_encoded['Суммарный объем очагов'].str.replace(',', '.').astype('float')

In [ ]:
df_encoded = df_encoded[(df_encoded['Время_метастазирования'].notna().astype(int) == 1) & (df_encoded['Время_реагирования'].notna().astype(int) == 1)]
df_encoded.drop(columns = ['Дистантные метастазы', 'Локальный рецидив'], axis = 1, inplace = True)

df_encoded['Объем на очаг'] = df_encoded['Суммарный объем очагов'] / df_encoded['Число очагов в ГМ']

#Для итоговых данных

df_test['Объем на очаг'] = df_test['Суммарный объем очагов'].astype('int') / df_test['Число очагов в ГМ']

In [ ]:
df_encoded['Прогрессия'].value_counts()
df_encoded = df_encoded[df_encoded['Время_реагирования'] >= 0]

df_test = df_test[df_test['Время_реагирования'] >= 0]


In [ ]:
df_encoded.drop(columns = ['Объем максимального очага'], axis = 1, inplace = True)
df_test.drop(columns = ['Объем максимального очага'], axis = 1, inplace = True)


In [ ]:
cat_cols = ['Онкологический диагноз', 'Лекарственное лечение']
for col in cat_cols:
    df_encoded[col] = df_encoded[col].fillna("Unknown")

for col in cat_cols:
    df_test[col] = df_test[col].fillna("Unknown")

### Новые признаки

## Генерация признаков
Временные интервалы из дат; скорости метастазирования/реагирования, индекс прогрессирования, объём на очаг, взаимодействие возраст×число очагов; винзоризация по 99-му перцентилю и `log1p` для скошенных признаков.

In [ ]:
df_encoded['Скорость_метастазирования'] = df_encoded['Суммарный объем очагов'] / df_encoded['Время_метастазирования']
df_encoded['Скорость_реагирования'] = df_encoded['Суммарный объем очагов'] / (df_encoded['Время_реагирования'] + 1)
df_encoded['Индекс_прогрессирования'] = df_encoded['Время_метастазирования'] - df_encoded['Время_реагирования']

df_test['Скорость_метастазирования'] = df_test['Суммарный объем очагов'] / df_test['Время_метастазирования']
df_test['Скорость_реагирования'] = df_test['Суммарный объем очагов'] / (df_test['Время_реагирования'] + 1)
df_test['Индекс_прогрессирования'] = df_test['Время_метастазирования'] - df_test['Время_реагирования']

In [ ]:
p_99_TM = df_encoded['Время_метастазирования'].quantile(0.99)
df_encoded['Время_метастазирования'] = df_encoded['Время_метастазирования'].apply(lambda x: 0 if x < 0 else (p_99_TM if x > p_99_TM else x))

p_99_TR = df_encoded['Время_реагирования'].quantile(0.99)
df_encoded['Время_реагирования'] = df_encoded['Время_реагирования'].apply(lambda x: 0 if x < 0 else (p_99_TR if x > p_99_TR else x))

p_99_SV = df_encoded['Суммарный объем очагов'].quantile(0.99)
df_encoded['Суммарный объем очагов'] = df_encoded['Суммарный объем очагов'].apply(lambda x: 0 if x < 0 else (p_99_SV if x > p_99_SV else x))

p_99_VM = df_encoded['Объем на очаг'].quantile(0.99)
df_encoded['Объем на очаг'] = df_encoded['Объем на очаг'].apply(lambda x: 0 if x < 0 else (p_99_VM if x > p_99_VM else x))

#Итоговые

p_99_TM = df_test['Время_метастазирования'].quantile(0.99)
df_test['Время_метастазирования'] = df_test['Время_метастазирования'].apply(lambda x: 0 if x < 0 else (p_99_TM if x > p_99_TM else x))

p_99_TR = df_test['Время_реагирования'].quantile(0.99)
df_test['Время_реагирования'] = df_test['Время_реагирования'].apply(lambda x: 0 if x < 0 else (p_99_TR if x > p_99_TR else x))

p_99_SV = df_test['Суммарный объем очагов'].quantile(0.99)
df_test['Суммарный объем очагов'] = df_test['Суммарный объем очагов'].apply(lambda x: 0 if x < 0 else (p_99_SV if x > p_99_SV else x))

p_99_VM = df_test['Объем на очаг'].quantile(0.99)
df_test['Объем на очаг'] = df_test['Объем на очаг'].apply(lambda x: 0 if x < 0 else (p_99_VM if x > p_99_VM else x))

In [ ]:
# ---- train: приведение и логарифмы ----
df_encoded['Суммарный объем очагов'] = (
    df_encoded['Суммарный объем очагов']
    .apply(lambda x: str(x).replace(',', '.'))
    .astype(float)
)

df_encoded['Суммарный объем очагов'] = np.log1p(df_encoded['Суммарный объем очагов'])
df_encoded['Объем на очаг'] = np.log1p(df_encoded['Объем на очаг'])
df_encoded['Время_метастазирования'] = np.log1p(df_encoded['Время_метастазирования'])
df_encoded['Время_реагирования'] = np.log1p(df_encoded['Время_реагирования'])

df_encoded['Возраст * число очагов'] = df_encoded['Возраст'] * df_encoded['Число очагов в ГМ']

# ---- test: приведение и логарифмы ----
df_test['Суммарный объем очагов'] = (
    df_test['Суммарный объем очагов']
    .apply(lambda x: str(x).replace(',', '.'))
    .astype(float)
)

df_test['Суммарный объем очагов'] = np.log1p(df_test['Суммарный объем очагов'])
df_test['Объем на очаг'] = np.log1p(df_test['Объем на очаг'])
df_test['Время_метастазирования'] = np.log1p(df_test['Время_метастазирования'])
df_test['Время_реагирования'] = np.log1p(df_test['Время_реагирования'])

df_test['Возраст * число очагов'] = df_test['Возраст'] * df_test['Число очагов в ГМ']

# ---- qcut: границы считаем по train, применяем к test ----
cats_train, bin_edges = pd.qcut(
    df_encoded["Суммарный объем очагов"],
    q=[0, 0.25, 0.5, 0.75, 1.0],
    retbins=True,
    duplicates='drop'
)

labels = list(range(len(bin_edges) - 1))

df_encoded["Категория"] = pd.cut(
    df_encoded["Суммарный объем очагов"],
    bins=bin_edges,
    labels=labels,
    include_lowest=True
)

df_test["Категория"] = pd.cut(
    df_test["Суммарный объем очагов"],
    bins=bin_edges,
    labels=labels,
    include_lowest=True
)

# Если в тесте появились NaN (значения вне диапазона или из-за краёв), прижмём:
min_label, max_label = labels[0], labels[-1]

mask_nan = df_test["Категория"].isna()
if mask_nan.any():
    df_test.loc[mask_nan & (df_test["Суммарный объем очагов"] <= bin_edges[0]), "Категория"] = min_label
    df_test.loc[mask_nan & (df_test["Суммарный объем очагов"] >= bin_edges[-1]), "Категория"] = max_label
    # если вдруг остались NaN (например, из-за NaN в объёме) — заполним минимальным
    df_test["Категория"] = df_test["Категория"].fillna(min_label)

# приводим к int
df_encoded["Категория"] = df_encoded["Категория"].astype(int)
df_test["Категория"] = df_test["Категория"].astype(int)

In [ ]:
df_encoded

# Разделение данных

In [ ]:
"""# Разделение данных"""

X = df_encoded.drop(columns=['Прогрессия'])
y = df_encoded['Прогрессия'].astype(int)

print("X shape:", X.shape, "| y mean:", y.mean())

# Подготовка данных

In [ ]:
"""# Подготовка данных под CatBoost и LightGBM"""

# --- CatBoost: категории строками ---
X_cb = X.copy()
test_cb = df_test.copy()

for c in cat_cols:
    X_cb[c] = X_cb[c].astype("object")
    test_cb[c] = test_cb[c].astype("object")

    X_cb[c] = X_cb[c].where(X_cb[c].notna(), "Unknown").astype(str)
    test_cb[c] = test_cb[c].where(test_cb[c].notna(), "Unknown").astype(str)

print("X_cb cat dtypes:")
print(X_cb[cat_cols].dtypes)


# Данные под Catboost и LightGBM

## Обучение моделей (OOF)
CatBoost с подбором гиперпараметров через **Optuna** (TPE), затем 5-fold OOF; аналогично LightGBM и XGBoost. Категории обрабатываются нативно.

In [ ]:
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import numpy as np
from catboost import CatBoostClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ускоренный CV для тюнинга
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# фикс индексов
X_cb = X_cb.reset_index(drop=True)
y = y.reset_index(drop=True)

def best_balanced_accuracy_from_oof(y_true, oof_probs):
    best = -1
    for t in np.linspace(0.6, 0.95, 200):
        best = max(best, balanced_accuracy_score(y_true, (oof_probs >= t).astype(int)))
    return best

def objective(trial):
    params = {
        "bootstrap_type": "Bernoulli",   # 🔥 быстрее и стабильнее
        "iterations": trial.suggest_int("iterations", 1800, 2800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.04, log=True),
        "depth": trial.suggest_int("depth", 7, 11),
        "l2_leaf_reg": trial.suggest_int("l2_leaf_reg", 5, 25),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 100),
        "random_strength": trial.suggest_float("random_strength", 0.5, 2.0),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 5.0, 10.0),

        "task_type": "GPU",
        "devices": "0",
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "random_seed": 42,
        "verbose": False,
    }

    oof = np.zeros(len(X_cb))

    for tr_idx, val_idx in CV.split(X_cb, y):
        model = CatBoostClassifier(**params)
        model.fit(
            X_cb.iloc[tr_idx], y.iloc[tr_idx],
            cat_features=cat_cols,
            eval_set=(X_cb.iloc[val_idx], y.iloc[val_idx]),
            early_stopping_rounds=100,
            verbose=False
        )
        oof[val_idx] = model.predict_proba(X_cb.iloc[val_idx])[:, 1]

    return best_balanced_accuracy_from_oof(y, oof)

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=25, show_progress_bar=True)

print("Best OOF BA:", study.best_value)
print("Best params:", study.best_params)


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
import numpy as np
from catboost import CatBoostClassifier

X_cb = X_cb.reset_index(drop=True)
y = y.reset_index(drop=True)

CV_FINAL = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_params = study.best_params.copy()
best_params["scale_pos_weight"] = 1.0


best_params.update({
    "bootstrap_type": "Bernoulli",
    "task_type": "GPU",
    "devices": "0",
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "random_seed": 42,
    "verbose": False,
    "od_type": "Iter",
})


oof_probs = np.zeros(len(X_cb), dtype=float)
models = []

for fold, (tr_idx, val_idx) in enumerate(CV_FINAL.split(X_cb, y), start=1):
    model = CatBoostClassifier(**best_params)
    model.fit(
        X_cb.iloc[tr_idx], y.iloc[tr_idx],
        cat_features=cat_cols,
        eval_set=(X_cb.iloc[val_idx], y.iloc[val_idx]),
        early_stopping_rounds=150,
        verbose=False
    )
    proba = model.predict_proba(X_cb.iloc[val_idx])[:, 1]
    oof_probs[val_idx] = proba
    models.append(model)
    print(f"Fold {fold} | AUC:", roc_auc_score(y.iloc[val_idx], proba))

best_th, best_bal = 0.5, -1.0
for t in np.linspace(0.05, 0.95, 181):
    bal = balanced_accuracy_score(y, (oof_probs >= t).astype(int))
    if bal > best_bal:
        best_bal, best_th = float(bal), float(t)

print("OOF AUC:", roc_auc_score(y, oof_probs))
print("Best threshold:", best_th)
print("Best OOF balanced accuracy:", best_bal)


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

# ====== 0) фиксируем, чтобы не путаться ======
oof_cb = oof_probs.copy()
models_cb = models  # твой список CatBoost моделей

# ====== 1) готовим данные под LGBM ======
X_lgb = X.copy()
for c in cat_cols:
    X_lgb[c] = X_lgb[c].astype("category")

X_lgb = X_lgb.reset_index(drop=True)
y_lgb = y.reset_index(drop=True)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_lgb), dtype=float)
models_lgb = []

# параметры можно слегка “ослабить”, чтобы не было тонны "No further splits..."
lgb_params = dict(
    n_estimators=4000,
    learning_rate=0.01,
    num_leaves=63,
    min_child_samples=10,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="binary",
    random_state=42
)

for fold, (tr_idx, val_idx) in enumerate(CV.split(X_lgb, y_lgb), start=1):
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(
        X_lgb.iloc[tr_idx], y_lgb.iloc[tr_idx],
        eval_set=[(X_lgb.iloc[val_idx], y_lgb.iloc[val_idx])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(200, verbose=False)]
    )
    oof_lgb[val_idx] = m.predict_proba(X_lgb.iloc[val_idx])[:, 1]
    models_lgb.append(m)

print("Corr(CB, LGBM) =", round(np.corrcoef(oof_cb, oof_lgb)[0,1], 4))

# ====== 2) подбираем лучший вес (alpha) и порог (threshold) под Balanced Accuracy ======
def best_alpha_th(y_true, p_cb, p_lgb):
    best = (-1.0, None, None)  # (score, alpha, th)
    for alpha in np.linspace(0.0, 1.0, 101):     # шаг 0.01
        p = alpha * p_cb + (1 - alpha) * p_lgb
        for th in np.linspace(0.05, 0.95, 181):  # шаг 0.005
            s = balanced_accuracy_score(y_true, (p >= th).astype(int))
            if s > best[0]:
                best = (float(s), float(alpha), float(th))
    return best

best_bal, best_alpha, best_th = best_alpha_th(y_lgb, oof_cb, oof_lgb)
print("Best OOF BA (blend):", best_bal)
print("Best alpha (CB weight):", best_alpha)
print("Best threshold:", best_th)

# ====== 3) предикт теста (CB + LGBM) и несколько сабмитов рядом с лучшим порогом ======
# test для CB (строки)
test_cb = df_test[X.columns].copy()
for c in cat_cols:
    test_cb[c] = test_cb[c].astype("object").where(test_cb[c].notna(), "Unknown").astype(str)
test_probs_cb = np.mean([m.predict_proba(test_cb)[:, 1] for m in models_cb], axis=0)

# test для LGBM (категории)
test_lgb = df_test[X.columns].copy()
for c in cat_cols:
    test_lgb[c] = test_lgb[c].fillna("Unknown").astype("category")
test_probs_lgb = np.mean([m.predict_proba(test_lgb)[:, 1] for m in models_lgb], axis=0)

test_probs_blend = best_alpha * test_probs_cb + (1 - best_alpha) * test_probs_lgb

# делаем несколько сабмитов вокруг лучшего порога
thresholds = [max(0.05, best_th - 0.04), best_th - 0.02, best_th, best_th + 0.02, min(0.95, best_th + 0.04)]
thresholds = [round(t, 3) for t in thresholds]

for t in thresholds:
    test_pred = (test_probs_blend >= t).astype(int)
    sub = pd.DataFrame({"ID": np.arange(1, len(test_pred)+1), "target": test_pred})
    fname = f"submission_blend_a{best_alpha:.2f}_t{t:.3f}.csv"
    sub.to_csv(fname, index=False)
    print("Saved:", fname, "| positives=", round(test_pred.mean(), 3))


## Блендинг: веса и порог
Перебор весов трёх моделей и порога классификации для максимизации balanced accuracy на OOF-предсказаниях.

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score

p_cb  = oof_probs.copy()
p_lgb = oof_probs_lgb.copy()
p_xgb = oof_probs_xgb.copy()
y_true = y.values if hasattr(y, "values") else np.asarray(y)

def search_weights_and_threshold(y_true, p_cb, p_lgb, p_xgb,
                                 step_w=0.02, th_grid=np.linspace(0.05, 0.95, 181)):
    best = (-1.0, None, None, None)  # (BA, w_cb, w_lgb, th)
    # w_cb + w_lgb + w_xgb = 1
    for w_cb in np.arange(0.0, 1.0 + 1e-9, step_w):
        for w_lgb in np.arange(0.0, 1.0 - w_cb + 1e-9, step_w):
            w_xgb = 1.0 - w_cb - w_lgb
            if w_xgb < -1e-9:
                continue
            p = w_cb*p_cb + w_lgb*p_lgb + w_xgb*p_xgb

            # лучший порог для этих весов
            # (можно ускорить, но так прозрачно и стабильно)
            for th in th_grid:
                pred = (p >= th).astype(int)
                ba = balanced_accuracy_score(y_true, pred)
                if ba > best[0]:
                    best = (float(ba), float(w_cb), float(w_lgb), float(th))
    return best

best_ba, w_cb, w_lgb, best_th = search_weights_and_threshold(
    y_true, p_cb, p_lgb, p_xgb,
    step_w=0.02,
    th_grid=np.linspace(0.50, 0.85, 351)  # можно сузить вокруг твоих 0.65-0.72
)

w_xgb = 1.0 - w_cb - w_lgb

print("Best OOF BA (triple blend):", best_ba)
print("Weights: w_cb =", round(w_cb,3), "| w_lgb =", round(w_lgb,3), "| w_xgb =", round(w_xgb,3))
print("Best threshold:", round(best_th,3))


In [ ]:
import numpy as np

test_cb = df_test[X.columns].copy()
for c in cat_cols:
    test_cb[c] = test_cb[c].astype("object").where(test_cb[c].notna(), "Unknown").astype(str)

test_probs_cb = np.mean([m.predict_proba(test_cb)[:,1] for m in models], axis=0)
print("test_probs_cb mean:", test_probs_cb.mean())


In [ ]:
test_lgb = df_test[X.columns].copy()
for c in cat_cols:
    test_lgb[c] = test_lgb[c].fillna("Unknown").astype("category")

test_probs_lgb = np.mean([m.predict_proba(test_lgb)[:,1] for m in models_lgb], axis=0)
print("test_probs_lgb mean:", test_probs_lgb.mean())


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X_xgb = X.copy()

# категории -> category
for c in cat_cols:
    X_xgb[c] = X_xgb[c].astype("category")

X_xgb = X_xgb.reset_index(drop=True)
y_xgb = y.reset_index(drop=True)

# === КЛЮЧ: убираем inf / -inf / очень большие ===
X_xgb = X_xgb.replace([np.inf, -np.inf], np.nan)

# опционально, но полезно: обрезать совсем дикие численные значения
num_cols = X_xgb.select_dtypes(include=["number"]).columns
X_xgb[num_cols] = X_xgb[num_cols].clip(-1e6, 1e6)

# контроль
bad = np.isinf(X_xgb[num_cols].to_numpy()).any() or np.isnan(X_xgb[num_cols].to_numpy()).sum() > 0
print("Has inf:", np.isinf(X_xgb[num_cols].to_numpy()).any())
print("NaN count:", np.isnan(X_xgb[num_cols].to_numpy()).sum())

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs_xgb = np.zeros(len(X_xgb), dtype=float)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 6,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "tree_method": "hist",   # вместо gpu_hist
    "device": "cuda",        # GPU (если нет — убери эту строку)
    "max_cat_to_onehot": 16,
    "random_state": 42,
}


for fold, (tr_idx, val_idx) in enumerate(CV.split(X_xgb, y_xgb), start=1):
    dtrain = xgb.DMatrix(
        X_xgb.iloc[tr_idx],
        label=y_xgb.iloc[tr_idx],
        enable_categorical=True,
        missing=np.nan
    )
    dval = xgb.DMatrix(
        X_xgb.iloc[val_idx],
        label=y_xgb.iloc[val_idx],
        enable_categorical=True,
        missing=np.nan
    )

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=800,
        evals=[(dval, "val")],
        verbose_eval=False
    )

    proba = model.predict(dval)
    oof_probs_xgb[val_idx] = proba
    print(f"Fold {fold} | XGB AUC:", roc_auc_score(y_xgb.iloc[val_idx], proba))

print("XGB oof mean:", oof_probs_xgb.mean())


In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold

# готовим XGB test
test_xgb = df_test[X.columns].copy()
for c in cat_cols:
    test_xgb[c] = test_xgb[c].fillna("Unknown").astype("category")

# same cleaning, как в train (на всякий)
test_xgb = test_xgb.replace([np.inf, -np.inf], np.nan)
num_cols = test_xgb.select_dtypes(include=["number"]).columns
test_xgb[num_cols] = test_xgb[num_cols].clip(-1e6, 1e6)

# если у тебя нет models_xgb — обучим 5 моделей и сохраним
models_xgb = []
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 6,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "device": "cuda",      # если на CPU — убери эту строку
    "max_cat_to_onehot": 16,
    "random_state": 42,
}

for tr_idx, val_idx in CV.split(X_xgb, y_xgb):
    dtrain = xgb.DMatrix(X_xgb.iloc[tr_idx], label=y_xgb.iloc[tr_idx], enable_categorical=True, missing=np.nan)
    dval   = xgb.DMatrix(X_xgb.iloc[val_idx], label=y_xgb.iloc[val_idx], enable_categorical=True, missing=np.nan)

    m = xgb.train(
        params,
        dtrain,
        num_boost_round=800,
        evals=[(dval, "val")],
        verbose_eval=False
    )
    models_xgb.append(m)

# test probs xgb
dtest = xgb.DMatrix(test_xgb, enable_categorical=True, missing=np.nan)
test_probs_xgb = np.mean([m.predict(dtest) for m in models_xgb], axis=0)
print("test_probs_xgb mean:", test_probs_xgb.mean())


## Диагностика и сабмиты
Корреляции OOF-предсказаний моделей и формирование финальных сабмитов вокруг лучшего порога.

In [ ]:
import numpy as np

print("Corr(CB, LGB) =", np.corrcoef(oof_probs, oof_probs_lgb)[0,1])
print("Corr(CB, XGB) =", np.corrcoef(oof_probs, oof_probs_xgb)[0,1])
print("Corr(LGB, XGB) =", np.corrcoef(oof_probs_lgb, oof_probs_xgb)[0,1])


In [ ]:
import pandas as pd
import numpy as np

test_probs_blend = w_cb*test_probs_cb + w_lgb*test_probs_lgb + (1.0-w_cb-w_lgb)*test_probs_xgb

# сделаем несколько сабмитов вокруг best_th
ths = [best_th-0.03, best_th-0.02, best_th-0.01, best_th, best_th+0.01, best_th+0.02, best_th+0.03]
ths = [float(np.clip(t, 0.05, 0.95)) for t in ths]

for t in ths:
    pred = (test_probs_blend >= t).astype(int)
    sub = pd.DataFrame({"ID": np.arange(1, len(pred)+1), "target": pred})
    fname = f"submission_triple_wcb{w_cb:.2f}_wlgb{w_lgb:.2f}_t{t:.3f}.csv"
    sub.to_csv(fname, index=False)
    print("Saved:", fname, "| positives=", round(pred.mean(), 3))
